In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skimage.io
from matplotlib.colors import ListedColormap
from skimage import exposure
from tifffile import imsave

%matplotlib inline
import cupy as cp
import cupyx.scipy.ndimage as ndimage
from attenuation.attenuation import attenuation_filter
from cupyx.scipy.ndimage import gaussian_filter, rotate
from tqdm import tqdm

cp.cuda.set_allocator(cp.cuda.MemoryPool(cp.cuda.malloc_managed).malloc)
import h5py

In [ ]:
import numpy as np
from matplotlib import pyplot as plt


def core_mask_creater(shapes, centers, radius, angles):
    center_x, center_y = centers
    shape_x, shape_y = shapes
    angle_1, angle_2 = angles
    radius = radius
    if angle_2 < angle_1:
        angle_2 += 360
    x, y = np.ogrid[-center_x : shape_x - center_x, -center_y : shape_y - center_y]
    mask = x * x + y * y <= radius * radius
    theta = np.rad2deg(np.arctan2(x, y))
    theta %= 360
    circle = np.zeros((shape_x, shape_y))
    circle[mask] = 1
    core_mask = theta <= (angle_2 - angle_1)
    return core_mask * circle

In [ ]:
# 1. multimosaic movie
positions = [10]
for position in positions:
    import os

    images_folder = (
        f"/20210503_201032_6_lines_mosaic_HB4_D4_processed/core_images_v17_06/"
    )
    if not os.path.exists(images_folder):
        os.makedirs(images_folder)
    stack_file = h5py.File(
        f"/20210503_201032_6_lines_mosaic_HB4_D4_processed/Position_{str(position)}_Settings_1_Processed/denoised_registered_processed.h5",
        "r",
    )
    cutout_wedge = [0.4, 0.4, 0.2, 0.2, 0.2, 0.2]
    cutout_wedge_angle = [30, 30, 30, 25, 20, 20]
    time_interval = 48
    all_mips = []
    for i in tqdm(range(6)):
        t = 1 + (time_interval) * i

        mempool = cp.get_default_memory_pool()
        pinned_mempool = cp.get_default_pinned_memory_pool()
        mempool.free_all_blocks()
        pinned_mempool.free_all_blocks()

        image = np.asarray(
            [
                stack_file[f"t{t:05}"]["s01"]["0"]["cells"][()].copy(),
                stack_file[f"t{t:05}"]["s00"]["0"]["cells"][()],
            ].copy()
        )

        image = image.astype(np.float32)

        down_scale_factor = 0.3
        image = ndimage.zoom(
            cp.asarray(image.astype(np.float32)),
            zoom=[
                1,
                down_scale_factor * (2 / 0.347),
                down_scale_factor * 1,
                down_scale_factor * 1,
            ],
            order=1,
        )

        mask = ~core_mask_creater(
            (image.shape[1], image.shape[3]),
            [int(475 * cutout_wedge[i]), int(689 * 0.5)],
            2000,
            (cutout_wedge_angle[i], 90),
        )
        cross_section_mask = np.zeros(image.shape[1:4])
        for i in range(image.shape[2]):
            cross_section_mask[:, i, :] = mask
        cross_section_mask = np.array(2 * [cross_section_mask])
        input_image = (cp.array(cross_section_mask) * image).get()
        angle = 160
        stack_mcherry_rotated = np.nan_to_num(
            rotate(
                cp.asarray(input_image[0]),
                angle,
                mode="constant",
                axes=(0, 2),
                reshape=True,
                order=0,
            )
        )
        print("rotated mcherry")
        stack_gfp_rotated = np.nan_to_num(
            rotate(
                cp.asarray(input_image[1]),
                angle,
                mode="constant",
                axes=(0, 2),
                reshape=True,
                order=0,
            )
        )
        print("rotated gfp")

        attenuation_filtering = 24
        attenuation_min_density = 0.3
        attenuation = 0.0005
        stack_mcherry_rotated = attenuation_filter(
            stack_mcherry_rotated,
            attenuation_min_density,
            attenuation,
            attenuation_filtering,
        ).get()
        print("attenuated mcherry")

        attenuation_filtering = 24
        attenuation_min_density = 0.3
        attenuation = 0.0005

        stack_gfp_rotated = attenuation_filter(
            stack_gfp_rotated,
            attenuation_min_density,
            attenuation,
            attenuation_filtering,
        ).get()
        print("attenuated gfp")

        plt.style.use("dark_background")
        max_proj = np.max(
            np.flip(np.asarray([stack_mcherry_rotated, stack_gfp_rotated]), axis=3),
            axis=1,
        )
        channels = ["GFP", "mCherry"]
        all_mips.append(max_proj)

    all_mips = np.array(all_mips)
    print("saving")
    imsave(
        images_folder + f"position_{position}_{24}h_core_MIP.tif",
        all_mips.astype(np.uint16),
        imagej=True,
        resolution=(
            1.0 / (0.347 / down_scale_factor),
            1.0 / (0.347 / down_scale_factor),
        ),
        metadata={"unit": "um", "axes": "TCYX"},
        compression="zlib",
    )

In [ ]:
all_stacked = np.hstack(np.moveaxis(all_mips[:, :, :, 110:790], 1, -1))
imsave(
    images_folder + f"grid_position_{position}_{24}h_core_MIP.tif",
    np.moveaxis(all_stacked, -1, 0).astype(np.uint16),
    imagej=True,
    resolution=(1.0 / (0.347 / down_scale_factor), 1.0 / (0.347 / down_scale_factor)),
    metadata={"unit": "um", "axes": "CYX"},
    compression="zlib",
)

In [ ]:
# 2. AGAR movie
positions = [15, 11]
# positions=[13,14,15,16]

all_mips = []
for position in positions:
    import os

    images_folder = f"/20220611_200804_ActGFP_LamRFP_TubRFFP_WTC_HB4_D4_pos1-4Mat_9-12_agar_processed/core_images_v17_06/"
    if not os.path.exists(images_folder):
        os.makedirs(images_folder)
    stack_file = h5py.File(
        f"/20220611_200804_ActGFP_LamRFP_TubRFFP_WTC_HB4_D4_pos1-4Mat_9-12_agar_processed/Position_{str(position)}_Settings_1_Processed/denoised_processed.h5",
        "r",
    )
    cutout_wedge = [0.2]
    cutout_wedge_angle = [20]
    time_interval = 24
    i = 0
    t = 1 + (time_interval) * 5

    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()

    image = np.asarray(
        [
            stack_file[f"t{t:05}"]["s01"]["0"]["cells"][()].copy(),
            stack_file[f"t{t:05}"]["s00"]["0"]["cells"][()],
        ].copy()
    )

    image = image.astype(np.float32)

    down_scale_factor = 0.3
    image = ndimage.zoom(
        cp.asarray(image.astype(np.float32)),
        zoom=[
            1,
            down_scale_factor * (2 / 0.347),
            down_scale_factor * 1,
            down_scale_factor * 1,
        ],
        order=1,
    )

    mask = ~core_mask_creater(
        (image.shape[1], image.shape[3]),
        [int(475 * cutout_wedge[i]), int(689 * 0.5)],
        2000,
        (cutout_wedge_angle[i], 90),
    )
    cross_section_mask = np.zeros(image.shape[1:4])
    for i in range(image.shape[2]):
        cross_section_mask[:, i, :] = mask
    cross_section_mask = np.array(2 * [cross_section_mask])
    input_image = (cp.array(cross_section_mask) * image).get()
    angle = 160
    stack_mcherry_rotated = np.nan_to_num(
        rotate(
            cp.asarray(input_image[0]),
            angle,
            mode="constant",
            axes=(0, 2),
            reshape=True,
            order=0,
        )
    )
    print("rotated mcherry")
    stack_gfp_rotated = np.nan_to_num(
        rotate(
            cp.asarray(input_image[1]),
            angle,
            mode="constant",
            axes=(0, 2),
            reshape=True,
            order=0,
        )
    )
    print("rotated gfp")

    attenuation_filtering = 24
    attenuation_min_density = 0.3
    attenuation = 0.0005
    stack_mcherry_rotated = attenuation_filter(
        stack_mcherry_rotated,
        attenuation_min_density,
        attenuation,
        attenuation_filtering,
    ).get()
    print("attenuated mcherry")

    attenuation_filtering = 24
    attenuation_min_density = 0.3
    attenuation = 0.0005

    stack_gfp_rotated = attenuation_filter(
        stack_gfp_rotated, attenuation_min_density, attenuation, attenuation_filtering
    ).get()
    print("attenuated gfp")

    plt.style.use("dark_background")
    im_proj = np.max(
        np.flip(np.asarray([stack_mcherry_rotated, stack_gfp_rotated]), axis=3), axis=1
    )
    im_proj = skimage.exposure.rescale_intensity(im_proj, out_range=(0, 1))
    green_map = [[0, i / 255, 0] for i in range(256)]
    green_matplotlib_map = ListedColormap(green_map, "Green")
    magenta_map = [[i / 255, 0, i / 255] for i in range(256)]
    magenta_matplotlib_map = ListedColormap(magenta_map, "Magenta")

    vmin_green, vmax_green = np.percentile(im_proj[0, :, :], q=(0.01, 99.9))
    clipped_green = exposure.rescale_intensity(
        im_proj[0, :, :], in_range=(vmin_green, vmax_green), out_range=np.float32
    )

    vmin_magenta, vmax_magenta = np.percentile(im_proj[1, :, :], q=(0.01, 99.9))
    clipped_magenta = exposure.rescale_intensity(
        im_proj[1, :, :], in_range=(vmin_magenta, vmax_magenta), out_range=np.float32
    )

    channel1 = green_matplotlib_map(clipped_green)
    channel2 = magenta_matplotlib_map(clipped_magenta)
    assembled = np.stack((channel2, channel1), axis=3)
    rgb_proj = np.max(assembled, axis=3)
    fig, axes = plt.subplots(1, figsize=(16, 16))
    axes.imshow(rgb_proj)
    axes.set_axis_off()
    fig.tight_layout()
    plt.show()
    max_proj = np.max(
        np.flip(np.asarray([stack_mcherry_rotated, stack_gfp_rotated]), axis=3), axis=1
    )
    channels = ["GFP", "mCherry"]
    all_mips.append(max_proj)


all_mips = np.array(all_mips)
print("saving")

In [ ]:
# 2. AGAR movie, matrigel
positions = [3]
# positions=[13,14,15,16]

all_mips = []
for position in positions:
    import os

    images_folder = f"/20220611_200804_ActGFP_LamRFP_TubRFFP_WTC_HB4_D4_pos1-4Mat_9-12_agar_processed/core_images_v13_08_24/"
    if not os.path.exists(images_folder):
        os.makedirs(images_folder)
    stack_file = h5py.File(
        f"/20220611_200804_ActGFP_LamRFP_TubRFFP_WTC_HB4_D4_pos1-4Mat_9-12_agar_processed/Position_{str(position)}_Settings_1_Processed/denoised_processed.h5",
        "r",
    )
    cutout_wedge = [0.2]
    cutout_wedge_angle = [20]
    time_interval = 24
    i = 0
    t = 1 + (time_interval) * 5

    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()

    image = np.asarray(
        [
            stack_file[f"t{t:05}"]["s01"]["0"]["cells"][()].copy(),
            stack_file[f"t{t:05}"]["s00"]["0"]["cells"][()],
        ].copy()
    )

    image = image.astype(np.float32)

    down_scale_factor = 0.3
    image = ndimage.zoom(
        cp.asarray(image.astype(np.float32)),
        zoom=[
            1,
            down_scale_factor * (2 / 0.347),
            down_scale_factor * 1,
            down_scale_factor * 1,
        ],
        order=1,
    )

    mask = ~core_mask_creater(
        (image.shape[1], image.shape[3]),
        [int(475 * cutout_wedge[i]), int(689 * 0.5)],
        2000,
        (cutout_wedge_angle[i], 90),
    )
    cross_section_mask = np.zeros(image.shape[1:4])
    for i in range(image.shape[2]):
        cross_section_mask[:, i, :] = mask
    cross_section_mask = np.array(2 * [cross_section_mask])
    input_image = (cp.array(cross_section_mask) * image).get()
    angle = 160
    stack_mcherry_rotated = np.nan_to_num(
        rotate(
            cp.asarray(input_image[0]),
            angle,
            mode="constant",
            axes=(0, 2),
            reshape=True,
            order=0,
        )
    )
    print("rotated mcherry")
    stack_gfp_rotated = np.nan_to_num(
        rotate(
            cp.asarray(input_image[1]),
            angle,
            mode="constant",
            axes=(0, 2),
            reshape=True,
            order=0,
        )
    )
    print("rotated gfp")

    attenuation_filtering = 24
    attenuation_min_density = 0.3
    attenuation = 0.0005
    stack_mcherry_rotated = attenuation_filter(
        stack_mcherry_rotated,
        attenuation_min_density,
        attenuation,
        attenuation_filtering,
    ).get()
    print("attenuated mcherry")

    attenuation_filtering = 24
    attenuation_min_density = 0.3
    attenuation = 0.0005

    stack_gfp_rotated = attenuation_filter(
        stack_gfp_rotated, attenuation_min_density, attenuation, attenuation_filtering
    ).get()
    print("attenuated gfp")

    plt.style.use("dark_background")
    im_proj = np.max(
        np.flip(np.asarray([stack_mcherry_rotated, stack_gfp_rotated]), axis=3), axis=1
    )
    im_proj = skimage.exposure.rescale_intensity(im_proj, out_range=(0, 1))
    green_map = [[0, i / 255, 0] for i in range(256)]
    green_matplotlib_map = ListedColormap(green_map, "Green")
    magenta_map = [[i / 255, 0, i / 255] for i in range(256)]
    magenta_matplotlib_map = ListedColormap(magenta_map, "Magenta")

    vmin_green, vmax_green = np.percentile(im_proj[0, :, :], q=(0.01, 99.9))
    clipped_green = exposure.rescale_intensity(
        im_proj[0, :, :], in_range=(vmin_green, vmax_green), out_range=np.float32
    )

    vmin_magenta, vmax_magenta = np.percentile(im_proj[1, :, :], q=(0.01, 99.9))
    clipped_magenta = exposure.rescale_intensity(
        im_proj[1, :, :], in_range=(vmin_magenta, vmax_magenta), out_range=np.float32
    )

    channel1 = green_matplotlib_map(clipped_green)
    channel2 = magenta_matplotlib_map(clipped_magenta)
    assembled = np.stack((channel2, channel1), axis=3)
    rgb_proj = np.max(assembled, axis=3)
    fig, axes = plt.subplots(1, figsize=(16, 16))
    axes.imshow(rgb_proj)
    axes.set_axis_off()
    fig.tight_layout()
    plt.show()
    max_proj = np.max(
        np.flip(np.asarray([stack_mcherry_rotated, stack_gfp_rotated]), axis=3), axis=1
    )
    channels = ["GFP", "mCherry"]
    all_mips.append(max_proj)


all_mips = np.array(all_mips)
print("saving")

In [ ]:
all_stacked = np.hstack(np.moveaxis(all_mips[:, :, 0:570, 100:690], 1, -1))

In [ ]:
all_mips.shape

In [ ]:
imsave(
    images_folder + f"grid_position_15_11_core_MIP.tif",
    np.moveaxis(all_stacked, -1, 0).astype(np.uint16),
    imagej=True,
    resolution=(1.0 / (0.347 / down_scale_factor), 1.0 / (0.347 / down_scale_factor)),
    metadata={"unit": "um", "axes": "CYX"},
    compression="zlib",
)